# 01 — Data Quality Assessment

## Olist E-Commerce Business Analytics

### Objective

This notebook evaluates the quality and reliability of the original Olist datasets before any cleaning or transformation is performed.

The assessment covers:

- Dataset dimensions and structure
- Column names and data types
- Missing values
- Exact duplicate rows
- Primary-key uniqueness
- Foreign-key relationships
- Categorical-value consistency
- Numerical-value validity
- Timestamp completeness and chronological logic

No source data will be modified in this notebook. All identified issues will be documented for the data-cleaning stage.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.3.3
NumPy version: 2.3.5


## 1. Locate the Raw Data

Before loading the datasets, we verify the notebook's working directory and locate the original CSV files.

In [9]:
DATA_DIR = Path("../data/raw").resolve()

print("Data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files:
    print("-", file.name)

Data directory: /Users/saadmaher/Desktop/Data Science/Project portfolio/olist-ecommerce-business-analytics/data/raw
Directory exists: True
CSV files found: 9
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


## 2. Dataset Inventory

The Olist data is distributed across nine relational CSV files. Each file represents a different business entity or process, including customers, orders, products, sellers, payments, reviews, and geographic information.

In [10]:
dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

print(f"Expected datasets: {len(dataset_files)}\n")

for dataset_name, filename in dataset_files.items():
    file_path = DATA_DIR / filename
    status = "Found" if file_path.exists() else "Missing"
    print(f"{dataset_name:<22} {status:<8} {filename}")

Expected datasets: 9

customers              Found    olist_customers_dataset.csv
geolocation            Found    olist_geolocation_dataset.csv
order_items            Found    olist_order_items_dataset.csv
payments               Found    olist_order_payments_dataset.csv
reviews                Found    olist_order_reviews_dataset.csv
orders                 Found    olist_orders_dataset.csv
products               Found    olist_products_dataset.csv
sellers                Found    olist_sellers_dataset.csv
category_translation   Found    product_category_name_translation.csv


## 3. Load the Raw Datasets

Each CSV file is loaded into a Pandas DataFrame and stored in a dictionary. Keeping the DataFrames in a dictionary makes it easier to apply the same data-quality checks consistently across all tables.

In [11]:
datasets = {}

for dataset_name, filename in dataset_files.items():
    file_path = DATA_DIR / filename
    datasets[dataset_name] = pd.read_csv(file_path)

print(f"Successfully loaded {len(datasets)} datasets.")

Successfully loaded 9 datasets.


### 3.1 Dataset Dimensions

The table below records the number of rows and columns in each raw dataset. Different row counts are expected because each table has a different grain. For example, the orders table contains one row per order, while the order-items table contains one row per item within an order.

In [12]:
for dataset_name, dataframe in datasets.items():
    rows, columns = dataframe.shape
    print(f"{dataset_name:<22} {rows:>10,} rows × {columns:>2} columns")

customers                  99,441 rows ×  5 columns
geolocation             1,000,163 rows ×  5 columns
order_items               112,650 rows ×  7 columns
payments                  103,886 rows ×  5 columns
reviews                    99,224 rows ×  7 columns
orders                     99,441 rows ×  8 columns
products                   32,951 rows ×  9 columns
sellers                     3,095 rows ×  4 columns
category_translation           71 rows ×  2 columns


In [13]:
inventory_records = []

for dataset_name, dataframe in datasets.items():
    inventory_records.append({
        "dataset": dataset_name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
        "memory_mb": dataframe.memory_usage(deep=True).sum() / (1024 ** 2)
    })

inventory_summary = (
    pd.DataFrame(inventory_records)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

inventory_summary

,dataset,rows,columns,memory_mb
0,geolocation,1000163,5,129.38
1,order_items,112650,7,35.99
2,payments,103886,5,16.23
3,customers,99441,5,26.59
4,orders,99441,8,52.94
5,reviews,99224,7,39.12
6,products,32951,9,6.30
7,sellers,3095,4,0.59
8,category_translation,71,2,0.01


***Observation:*** The project contains nine relational datasets with substantially different sizes. The geolocation table is the largest, with more than one million rows, while the category translation table contains only 71 records.

***Why it matters:*** The different table sizes reflect different levels of detail. Orders, order items, payments, and reviews must not be joined without first understanding their grain, because one-to-many relationships could duplicate records and inflate business metrics.

### 3.2 Schema Overview

A schema describes the structure of each dataset, including column names, detected data types, non-null values, missing values, and distinct values. This assessment helps identify incorrect data types, incomplete columns, identifiers, and categorical variables.

In [14]:
schema_records = []

for dataset_name, dataframe in datasets.items():
    for column in dataframe.columns:
        schema_records.append({
            "dataset": dataset_name,
            "column": column,
            "data_type": str(dataframe[column].dtype),
            "non_null_count": dataframe[column].notna().sum(),
            "missing_count": dataframe[column].isna().sum(),
            "unique_values": dataframe[column].nunique(dropna=True)
        })

schema_summary = pd.DataFrame(schema_records)

schema_summary

,dataset,column,data_type,non_null_count,missing_count,unique_values
0,customers,customer_id,object,99441,0,99441
1,customers,customer_unique_id,object,99441,0,96096
2,customers,customer_zip_code_prefix,int64,99441,0,14994
3,customers,customer_city,object,99441,0,4119
4,customers,customer_state,object,99441,0,27
5,geolocation,geolocation_zip_code_prefix,int64,1000163,0,19015
6,geolocation,geolocation_lat,float64,1000163,0,717363
7,geolocation,geolocation_lng,float64,1000163,0,717615
8,geolocation,geolocation_city,object,1000163,0,8011
9,geolocation,geolocation_state,object,1000163,0,27


In [15]:
print(f"Datasets documented: {schema_summary['dataset'].nunique()}")
print(f"Columns documented: {len(schema_summary)}")

Datasets documented: 9
Columns documented: 52


***Observation:*** The nine datasets contain 52 columns covering customer identifiers, orders, products, sellers, payments, reviews, locations, and timestamps. Identifier and text columns were imported as objects, while quantities and monetary values were generally detected as numerical types.

***Why it matters:*** Timestamp columns are initially stored as text and will require explicit date conversion during data cleaning. Identifier columns must remain categorical identifiers rather than being treated as numerical measurements.

## 4. Missing-Value Assessment

Missing values can affect joins, calculations, delivery metrics, product analysis, and customer-satisfaction analysis. Missingness must be measured before deciding whether values should be retained, imputed, excluded, or documented as a limitation.

In [16]:
missing_records = []

for dataset_name, dataframe in datasets.items():
    for column in dataframe.columns:
        missing_count = dataframe[column].isna().sum()
        missing_rate = missing_count / len(dataframe)

        missing_records.append({
            "dataset": dataset_name,
            "column": column,
            "missing_count": missing_count,
            "missing_rate": missing_rate
        })

missing_summary = (
    pd.DataFrame(missing_records)
    .query("missing_count > 0")
    .sort_values(
        ["missing_rate", "missing_count"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

missing_summary.style.format({
    "missing_count": "{:,.0f}",
    "missing_rate": "{:.2%}"
})

,dataset,column,missing_count,missing_rate
0,reviews,review_comment_title,"87,656",88.34%
1,reviews,review_comment_message,"58,247",58.70%
2,orders,order_delivered_customer_date,"2,965",2.98%
3,products,product_category_name,610,1.85%
4,products,product_name_lenght,610,1.85%
5,products,product_description_lenght,610,1.85%
6,products,product_photos_qty,610,1.85%
7,orders,order_delivered_carrier_date,"1,783",1.79%
8,orders,order_approved_at,160,0.16%
9,products,product_weight_g,2,0.01%


***Observation:*** Missing values are concentrated in optional review comments, order delivery timestamps, and a small group of product attributes. Review text is absent for most reviews, while missing delivery dates may be explained by orders that were cancelled, unavailable, or not yet delivered.

***Decision:*** No records will be removed at this stage. Missing review comments will be treated as “no written comment,” while order timestamps and product attributes require further investigation before a cleaning decision is made.

## 5. Duplicate-Row Assessment

Exact duplicate rows can inflate counts, revenue, order volumes, and other metrics. This check identifies records where every column contains the same value as another row in the same dataset.

In [17]:
duplicate_records = []

for dataset_name, dataframe in datasets.items():
    duplicate_count = dataframe.duplicated().sum()
    duplicate_rate = duplicate_count / len(dataframe)

    duplicate_records.append({
        "dataset": dataset_name,
        "total_rows": len(dataframe),
        "duplicate_rows": duplicate_count,
        "duplicate_rate": duplicate_rate
    })

duplicate_summary = (
    pd.DataFrame(duplicate_records)
    .sort_values("duplicate_rows", ascending=False)
    .reset_index(drop=True)
)

duplicate_summary.style.format({
    "total_rows": "{:,.0f}",
    "duplicate_rows": "{:,.0f}",
    "duplicate_rate": "{:.2%}"
})

,dataset,total_rows,duplicate_rows,duplicate_rate
0,geolocation,"1,000,163","261,831",26.18%
1,customers,"99,441",0,0.00%
2,order_items,"112,650",0,0.00%
3,payments,"103,886",0,0.00%
4,reviews,"99,224",0,0.00%
5,orders,"99,441",0,0.00%
6,products,"32,951",0,0.00%
7,sellers,"3,095",0,0.00%
8,category_translation,71,0,0.00%


***Observation:*** Eight of the nine datasets contain no exact duplicate rows. The geolocation dataset contains 261,831 duplicated rows, representing 26.18% of its records.

***Decision:*** The geolocation duplicates will be investigated and handled during data cleaning. They should not affect transaction metrics, but retaining them could increase processing time and distort geographic calculations if coordinates are counted rather than aggregated appropriately.

## 6. Primary-Key Validation

Primary keys uniquely identify records within a table. This assessment checks whether the expected keys contain missing values or duplicate combinations that could compromise joins and aggregations.

In [18]:
primary_key_definitions = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"]
}

primary_key_records = []

for dataset_name, key_columns in primary_key_definitions.items():
    dataframe = datasets[dataset_name]

    missing_key_rows = dataframe[key_columns].isna().any(axis=1).sum()
    duplicate_key_rows = dataframe.duplicated(
        subset=key_columns,
        keep=False
    ).sum()

    primary_key_records.append({
        "dataset": dataset_name,
        "primary_key": " + ".join(key_columns),
        "total_rows": len(dataframe),
        "missing_key_rows": missing_key_rows,
        "duplicate_key_rows": duplicate_key_rows,
        "key_status": (
            "Valid"
            if missing_key_rows == 0 and duplicate_key_rows == 0
            else "Requires investigation"
        )
    })

primary_key_summary = pd.DataFrame(primary_key_records)

primary_key_summary.style.format({
    "total_rows": "{:,.0f}",
    "missing_key_rows": "{:,.0f}",
    "duplicate_key_rows": "{:,.0f}"
})

,dataset,primary_key,total_rows,missing_key_rows,duplicate_key_rows,key_status
0,customers,customer_id,"99,441",0,0,Valid
1,orders,order_id,"99,441",0,0,Valid
2,order_items,order_id + order_item_id,"112,650",0,0,Valid
3,payments,order_id + payment_sequential,"103,886",0,0,Valid
4,reviews,review_id + order_id,"99,224",0,0,Valid
5,products,product_id,"32,951",0,0,Valid
6,sellers,seller_id,"3,095",0,0,Valid
7,category_translation,product_category_name,71,0,0,Valid


***Observation:*** All expected primary and composite keys contain no missing values or duplicate combinations. The core transactional tables therefore have reliable record-level identifiers for integration and analysis.

***Limitation:*** The geolocation dataset has no natural unique primary key and was excluded from this validation. Its ZIP-code prefixes can appear multiple times because a single prefix may be associated with several coordinate records.

## 7. Referential-Integrity Assessment

Referential integrity ensures that identifiers in one table have valid matching records in the related table. Broken relationships can cause records to disappear during joins and produce incomplete business metrics.

In [19]:
foreign_key_definitions = [
    {
        "relationship": "Orders → Customers",
        "child_dataset": "orders",
        "child_key": "customer_id",
        "parent_dataset": "customers",
        "parent_key": "customer_id"
    },
    {
        "relationship": "Order Items → Orders",
        "child_dataset": "order_items",
        "child_key": "order_id",
        "parent_dataset": "orders",
        "parent_key": "order_id"
    },
    {
        "relationship": "Payments → Orders",
        "child_dataset": "payments",
        "child_key": "order_id",
        "parent_dataset": "orders",
        "parent_key": "order_id"
    },
    {
        "relationship": "Reviews → Orders",
        "child_dataset": "reviews",
        "child_key": "order_id",
        "parent_dataset": "orders",
        "parent_key": "order_id"
    },
    {
        "relationship": "Order Items → Products",
        "child_dataset": "order_items",
        "child_key": "product_id",
        "parent_dataset": "products",
        "parent_key": "product_id"
    },
    {
        "relationship": "Order Items → Sellers",
        "child_dataset": "order_items",
        "child_key": "seller_id",
        "parent_dataset": "sellers",
        "parent_key": "seller_id"
    },
    {
        "relationship": "Products → Category Translation",
        "child_dataset": "products",
        "child_key": "product_category_name",
        "parent_dataset": "category_translation",
        "parent_key": "product_category_name"
    }
]

foreign_key_records = []

for relationship in foreign_key_definitions:
    child_values = datasets[
        relationship["child_dataset"]
    ][relationship["child_key"]].dropna()

    parent_values = datasets[
        relationship["parent_dataset"]
    ][relationship["parent_key"]].dropna()

    unmatched_mask = ~child_values.isin(parent_values)

    foreign_key_records.append({
        "relationship": relationship["relationship"],
        "child_rows_checked": len(child_values),
        "unmatched_rows": unmatched_mask.sum(),
        "unmatched_unique_values": child_values[unmatched_mask].nunique(),
        "match_rate": 1 - unmatched_mask.mean()
    })

foreign_key_summary = pd.DataFrame(foreign_key_records)

foreign_key_summary.style.format({
    "child_rows_checked": "{:,.0f}",
    "unmatched_rows": "{:,.0f}",
    "unmatched_unique_values": "{:,.0f}",
    "match_rate": "{:.2%}"
})

,relationship,child_rows_checked,unmatched_rows,unmatched_unique_values,match_rate
0,Orders → Customers,"99,441",0,0,100.00%
1,Order Items → Orders,"112,650",0,0,100.00%
2,Payments → Orders,"103,886",0,0,100.00%
3,Reviews → Orders,"99,224",0,0,100.00%
4,Order Items → Products,"112,650",0,0,100.00%
5,Order Items → Sellers,"112,650",0,0,100.00%
6,Products → Category Translation,"32,341",13,2,99.96%


***Observation:*** All core transactional foreign-key relationships achieved a 100% match rate, confirming that customers, orders, items, payments, reviews, products, and sellers can be integrated safely. The category translation table does not cover 13 products belonging to two Portuguese product categories, producing a minor 0.04% translation gap.

***Decision:*** The two untranslated categories will be identified and assigned documented English labels during data cleaning rather than excluding the affected products.

In [20]:
product_categories = datasets["products"]["product_category_name"].dropna()
translated_categories = set(
    datasets["category_translation"]["product_category_name"].dropna()
)

untranslated_categories = sorted(
    set(product_categories) - translated_categories
)

untranslated_product_count = (
    product_categories.isin(untranslated_categories).sum()
)

print("Untranslated categories:", len(untranslated_categories))
print("Affected products:", untranslated_product_count)

for category in untranslated_categories:
    print("-", category)

Untranslated categories: 2
Affected products: 13
- pc_gamer
- portateis_cozinha_e_preparadores_de_alimentos


### 7.1 Order Relationship Coverage

A valid foreign key does not guarantee that every order has a corresponding item, payment, or review. This assessment measures the completeness of each order-level relationship.

In [21]:
order_ids = datasets["orders"]["order_id"]

order_coverage_records = []

related_tables = {
    "order_items": datasets["order_items"]["order_id"],
    "payments": datasets["payments"]["order_id"],
    "reviews": datasets["reviews"]["order_id"]
}

for related_dataset, related_order_ids in related_tables.items():
    has_related_record = order_ids.isin(related_order_ids)

    order_coverage_records.append({
        "related_dataset": related_dataset,
        "total_orders": len(order_ids),
        "orders_with_record": has_related_record.sum(),
        "orders_without_record": (~has_related_record).sum(),
        "coverage_rate": has_related_record.mean()
    })

order_coverage_summary = pd.DataFrame(order_coverage_records)

order_coverage_summary.style.format({
    "total_orders": "{:,.0f}",
    "orders_with_record": "{:,.0f}",
    "orders_without_record": "{:,.0f}",
    "coverage_rate": "{:.2%}"
})

,related_dataset,total_orders,orders_with_record,orders_without_record,coverage_rate
0,order_items,"99,441","98,666",775,99.22%
1,payments,"99,441","99,440",1,100.00%
2,reviews,"99,441","98,673",768,99.23%


***Observation:*** Most orders have corresponding item, payment, and review records. However, 775 orders have no order-item record, one order has no payment record, and 768 orders have no review.

***Analytical impact:*** Orders without items cannot contribute to product, seller, or item-price analysis, while orders without reviews cannot contribute to satisfaction analysis. Their order statuses must be investigated before deciding whether these gaps represent data-quality problems or expected incomplete and cancelled transactions.

## 8. Order-Status Assessment

Order status describes the stage reached by each transaction. Understanding the status distribution is necessary before analysing revenue, delivery, satisfaction, and missing order-related records.

In [22]:
order_status_summary = (
    datasets["orders"]["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

order_status_summary["order_rate"] = (
    order_status_summary["order_count"]
    / order_status_summary["order_count"].sum()
)

order_status_summary.style.format({
    "order_count": "{:,.0f}",
    "order_rate": "{:.2%}"
})

,order_status,order_count,order_rate
0,delivered,"96,478",97.02%
1,shipped,"1,107",1.11%
2,canceled,625,0.63%
3,unavailable,609,0.61%
4,invoiced,314,0.32%
5,processing,301,0.30%
6,created,5,0.01%
7,approved,2,0.00%


***Observation:*** Delivered orders represent 97.02% of all transactions, while the remaining 2.98% are shipped, cancelled, unavailable, invoiced, processing, created, or approved. The incomplete order statuses likely explain many missing carrier and customer delivery timestamps.

***Analytical decision:*** Delivery-performance metrics will use delivered orders with valid delivery timestamps. Other analyses will retain relevant non-delivered statuses when necessary, particularly when examining cancellations, unavailable products, and order completion.

### 8.1 Missing Records by Order Status

Missing timestamps and related records may be expected for orders that were cancelled, unavailable, or had not completed the delivery process. This assessment groups data-quality gaps by order status to determine whether the missingness follows the business lifecycle.

In [23]:
orders_quality = datasets["orders"].copy()

orders_quality["missing_approval_date"] = (
    orders_quality["order_approved_at"].isna()
)

orders_quality["missing_carrier_date"] = (
    orders_quality["order_delivered_carrier_date"].isna()
)

orders_quality["missing_customer_date"] = (
    orders_quality["order_delivered_customer_date"].isna()
)

orders_quality["missing_order_items"] = (
    ~orders_quality["order_id"].isin(
        datasets["order_items"]["order_id"]
    )
)

orders_quality["missing_payment"] = (
    ~orders_quality["order_id"].isin(
        datasets["payments"]["order_id"]
    )
)

orders_quality["missing_review"] = (
    ~orders_quality["order_id"].isin(
        datasets["reviews"]["order_id"]
    )
)

status_quality_summary = (
    orders_quality
    .groupby("order_status", as_index=False)
    .agg(
        total_orders=("order_id", "size"),
        missing_approval_date=("missing_approval_date", "sum"),
        missing_carrier_date=("missing_carrier_date", "sum"),
        missing_customer_date=("missing_customer_date", "sum"),
        missing_order_items=("missing_order_items", "sum"),
        missing_payment=("missing_payment", "sum"),
        missing_review=("missing_review", "sum")
    )
    .sort_values("total_orders", ascending=False)
    .reset_index(drop=True)
)

status_quality_summary.style.format({
    column: "{:,.0f}"
    for column in status_quality_summary.columns
    if column != "order_status"
})

,order_status,total_orders,missing_approval_date,missing_carrier_date,missing_customer_date,missing_order_items,missing_payment,missing_review
0,delivered,"96,478",14,2,8,0,1,646
1,shipped,"1,107",0,0,"1,107",1,0,75
2,canceled,625,141,550,619,164,0,20
3,unavailable,609,0,609,609,603,0,14
4,invoiced,314,0,314,314,2,0,5
5,processing,301,0,301,301,0,0,6
6,created,5,5,5,5,5,0,2
7,approved,2,0,2,2,0,0,0


***Observation:*** Missing delivery timestamps are strongly associated with incomplete order statuses. Shipped orders have no customer delivery date, while cancelled, unavailable, invoiced, processing, created, and approved orders generally did not reach the carrier or customer delivery stages.

***Data-quality concern:*** A small number of delivered orders still contain unexpected gaps: 14 lack an approval date, two lack a carrier delivery date, eight lack a customer delivery date, and one lacks a payment record. Additionally, 646 delivered orders have no review, although submitting a review is optional.

***Analytical decision:*** Expected lifecycle-related missing values will be retained and interpreted according to order status. Delivered orders with missing timestamps will be flagged for investigation and excluded only from calculations requiring those specific dates.

## 9. Date Validity Assessment

Order timestamps should follow the expected business sequence from purchase to approval, carrier handoff, and customer delivery. Dates are converted in a temporary copy so the raw datasets remain unchanged.

In [24]:
orders_dates = datasets["orders"].copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_dates[column] = pd.to_datetime(
        orders_dates[column],
        errors="coerce"
    )

date_quality_summary = pd.DataFrame({
    "check": [
        "Approval before purchase",
        "Carrier handoff before purchase",
        "Customer delivery before purchase",
        "Customer delivery before carrier handoff",
        "Delivered after estimated date"
    ],
    "affected_orders": [
        (
            orders_dates["order_approved_at"]
            < orders_dates["order_purchase_timestamp"]
        ).sum(),
        (
            orders_dates["order_delivered_carrier_date"]
            < orders_dates["order_purchase_timestamp"]
        ).sum(),
        (
            orders_dates["order_delivered_customer_date"]
            < orders_dates["order_purchase_timestamp"]
        ).sum(),
        (
            orders_dates["order_delivered_customer_date"]
            < orders_dates["order_delivered_carrier_date"]
        ).sum(),
        (
            orders_dates["order_delivered_customer_date"]
            > orders_dates["order_estimated_delivery_date"]
        ).sum()
    ]
})

date_quality_summary["affected_rate"] = (
    date_quality_summary["affected_orders"]
    / len(orders_dates)
)

date_quality_summary.style.format({
    "affected_orders": "{:,.0f}",
    "affected_rate": "{:.2%}"
})

,check,affected_orders,affected_rate
0,Approval before purchase,0,0.00%
1,Carrier handoff before purchase,166,0.17%
2,Customer delivery before purchase,0,0.00%
3,Customer delivery before carrier handoff,23,0.02%
4,Delivered after estimated date,"7,827",7.87%


***Observation:*** No orders were approved or delivered to customers before purchase. However, 166 orders were recorded as handed to the carrier before purchase, and 23 were recorded as delivered to the customer before carrier handoff, indicating a small number of timestamp inconsistencies.

***Business finding:*** A total of 7,827 orders were delivered after their estimated delivery date. This represents a delivery-performance issue rather than a data-quality error and will be investigated during the delivery analytics stage.

***Analytical decision:*** The 189 potentially inconsistent timestamp sequences will be flagged during cleaning and excluded from calculations where they would produce invalid delivery durations. Late deliveries will be retained because they represent valid and important business outcomes.

## 10. Numerical and Categorical Validity

This assessment checks whether important monetary, rating, product, payment, and order-status values fall within reasonable business ranges.

In [25]:
order_items = datasets["order_items"]
payments = datasets["payments"]
products = datasets["products"]
reviews = datasets["reviews"]
orders = datasets["orders"]

expected_order_statuses = {
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved"
}

validity_checks = [
    {
        "check": "Negative item prices",
        "affected_rows": (order_items["price"] < 0).sum()
    },
    {
        "check": "Negative freight values",
        "affected_rows": (order_items["freight_value"] < 0).sum()
    },
    {
        "check": "Negative payment values",
        "affected_rows": (payments["payment_value"] < 0).sum()
    },
    {
        "check": "Zero payment values",
        "affected_rows": (payments["payment_value"] == 0).sum()
    },
    {
        "check": "Non-positive product weights",
        "affected_rows": (
            products["product_weight_g"].notna()
            & (products["product_weight_g"] <= 0)
        ).sum()
    },
    {
        "check": "Non-positive product dimensions",
        "affected_rows": (
            (
                products[
                    [
                        "product_length_cm",
                        "product_height_cm",
                        "product_width_cm"
                    ]
                ] <= 0
            ).any(axis=1)
        ).sum()
    },
    {
        "check": "Review scores outside 1–5",
        "affected_rows": (
            ~reviews["review_score"].between(1, 5)
        ).sum()
    },
    {
        "check": "Unexpected order statuses",
        "affected_rows": (
            ~orders["order_status"].isin(expected_order_statuses)
        ).sum()
    },
    {
        "check": "Zero payment installments",
        "affected_rows": (
            payments["payment_installments"] == 0
        ).sum()
    },
    {
        "check": "Undefined payment types",
        "affected_rows": (
            payments["payment_type"] == "not_defined"
        ).sum()
    }
]

validity_summary = pd.DataFrame(validity_checks)

validity_summary.style.format({
    "affected_rows": "{:,.0f}"
})

,check,affected_rows
0,Negative item prices,0
1,Negative freight values,0
2,Negative payment values,0
3,Zero payment values,9
4,Non-positive product weights,4
5,Non-positive product dimensions,0
6,Review scores outside 1–5,0
7,Unexpected order statuses,0
8,Zero payment installments,2
9,Undefined payment types,3


***Observation:*** No negative prices, freight values, or payment values were found. All review scores fall within the expected 1–5 range, all order statuses are recognised, and no non-positive product dimensions were detected.

***Data-quality concern:*** Nine payment records have a value of zero, four products have non-positive weights, two payment records report zero installments, and three payment methods are labelled `not_defined`.

***Analytical decision:*** These low-frequency anomalies will be investigated during cleaning rather than removed automatically. Zero-value payments may represent voucher or processing behaviour, while non-positive weights could affect logistics analysis and require null treatment or exclusion from weight-based calculations.

## 11. Final Data-Quality Summary

The following report consolidates the principal findings, their analytical significance, and the actions planned for the data-cleaning stage.

In [27]:
data_quality_report.loc[0, "issue"] = "Missing review titles"

missing_message_row = pd.DataFrame([{
    "issue": "Missing review messages",
    "affected_records": 58247,
    "severity": "Low",
    "planned_action": (
        "Retain missing values and distinguish ratings "
        "with no written feedback."
    )
}])

data_quality_report = pd.concat(
    [
        data_quality_report.iloc[:1],
        missing_message_row,
        data_quality_report.iloc[1:]
    ],
    ignore_index=True
)

data_quality_report

,issue,affected_records,severity,planned_action
0,Missing review titles,87656,Low,Retain missing values and distinguish ratings ...
1,Missing review messages,58247,Low,Retain missing values and distinguish ratings ...
2,Missing order lifecycle timestamps,2965,Medium,Interpret according to order status and flag u...
3,Missing product categories and descriptive att...,610,Medium,Assign an Unknown category where appropriate a...
4,Exact duplicate geolocation rows,261831,Medium,Remove exact duplicates from the cleaned geolo...
5,Untranslated product categories,13,Low,Add documented English translations for the tw...
6,Orders without order-item records,775,Medium,Retain for order-status analysis but exclude f...
7,Orders without payment records,1,Low,Investigate and exclude only from payment-depe...
8,Orders without review records,768,Low,Retain orders and exclude only from review-dep...
9,Carrier handoff before purchase,166,Medium,Flag and exclude from calculations producing i...


In [28]:
OUTPUT_DIR = Path("../data/processed").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_path = OUTPUT_DIR / "data_quality_report.csv"
data_quality_report.to_csv(report_path, index=False)

print("Report saved to:")
print(report_path)

Report saved to:
/Users/saadmaher/Desktop/Data Science/Project portfolio/olist-ecommerce-business-analytics/data/processed/data_quality_report.csv


## Conclusion

***Overall assessment:*** The Olist datasets are suitable for the planned business analysis. Primary keys are complete, core foreign-key relationships are valid, monetary values are generally reasonable, and most missing timestamps are explained by the order lifecycle.

***Main limitations:*** The principal issues are duplicated geolocation records, missing optional review text, incomplete product attributes, a small translation gap, and a limited number of timestamp and payment anomalies.

***Next step:*** The identified issues will be handled transparently in `02_data_cleaning.ipynb`. The original CSV files will remain unchanged, and every transformation will be applied only to cleaned copies.